In [0]:
from pyspark.sql import functions as F

df_silver = spark.sql("SELECT * FROM files.amazon_files.amazon_silver")

In [0]:
df_preco_gold = df_silver.withColumn("categoria_principal", F.split(F.col("category"), "\|")[0])

<>:1: SyntaxWarning: invalid escape sequence '\|'
<>:1: SyntaxWarning: invalid escape sequence '\|'
/home/spark-175ec0b1-3c7d-4e04-a300-dc/.ipykernel/89/command-6654444828940655-863674221:1: SyntaxWarning: invalid escape sequence '\|'
  df_preco_gold = df_silver.withColumn("categoria_principal", F.split(F.col("category"), "\|")[0])


In [0]:
df_preco_gold.select("category", "categoria_principal").show(10, truncate=False)

+---------------------------------------------------------------------------------+---------------------+
|category                                                                         |categoria_principal  |
+---------------------------------------------------------------------------------+---------------------+
|Computers&Accessories|Accessories&Peripherals|Cables&Accessories|Cables|USBCables|Computers&Accessories|
|Computers&Accessories|Accessories&Peripherals|Cables&Accessories|Cables|USBCables|Computers&Accessories|
|Computers&Accessories|Accessories&Peripherals|Cables&Accessories|Cables|USBCables|Computers&Accessories|
|Computers&Accessories|Accessories&Peripherals|Cables&Accessories|Cables|USBCables|Computers&Accessories|
|Computers&Accessories|Accessories&Peripherals|Cables&Accessories|Cables|USBCables|Computers&Accessories|
|Computers&Accessories|Accessories&Peripherals|Cables&Accessories|Cables|USBCables|Computers&Accessories|
|Computers&Accessories|Accessories&Peripherals

In [0]:
df_gold_preco = df_preco_gold.groupBy("categoria_principal") \
    .agg(
        F.count("*").alias("qtd_produtos"),
        F.expr("percentile_approx(actual_price, 0.5)").alias("mediana_preco"),
        F.min("actual_price").alias("preco_minimo"),
        F.max("actual_price").alias("preco_maximo")
    ) \
        .orderBy(F.desc("mediana_preco"))

df_gold_preco.show()

+--------------------+------------+-------------+------------+------------+
| categoria_principal|qtd_produtos|mediana_preco|preco_minimo|preco_maximo|
+--------------------+------------+-------------+------------+------------+
|       Car&Motorbike|           1|       4000.0|      4000.0|      4000.0|
|         Electronics|         476|       2499.0|       171.0|    139900.0|
|        Home&Kitchen|         394|       1999.0|        89.0|     75990.0|
|Computers&Accesso...|         447|        999.0|        39.0|     37999.0|
|  MusicalInstruments|           2|        699.0|       699.0|      1995.0|
|     HomeImprovement|           2|        599.0|       599.0|       999.0|
|      OfficeProducts|          30|        210.0|        50.0|      2999.0|
|          Toys&Games|           1|        150.0|       150.0|       150.0|
+--------------------+------------+-------------+------------+------------+



In [0]:
df_gold_preco.write.format("delta").mode("overwrite").saveAsTable("files.amazon_files.amazon_gold_preco")